In [ ]:
from typing import TypedDict

class Person(TypedDict):
    name: str
    age: int

In [3]:
new_person: Person = {"name": "Tanish", "age": 21}

In [4]:
print(new_person)

{'name': 'Tanish', 'age': 21}


SENTIMENT ANALYSIS

In [12]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()
model = init_chat_model("google_genai:gemini-2.5-flash-lite")

In [13]:
class Review(TypedDict):
    summary: str
    sentiment: str

In [14]:
structured_model = model.with_structured_output(Review)

In [20]:
result = structured_model.invoke("""The hardware is great, but the software feels bloated. There are too many pre-installed apps that i can't remove.""")

In [21]:
print(result)

{'summary': 'The hardware is good, but the software is bloated with unremovable pre-installed apps.', 'sentiment': 'negative'}


ANNOTATED

In [23]:
from typing import TypedDict, Annotated, Optional, Literal

# schema
class Review(TypedDict):

    key_themes: Annotated[list[str], "Write down all the key themes discussed in the review in a list"]
    summary: Annotated[str, "A brief summary of the review"]
    sentiment: Annotated[Literal["pos", "neg"], "Return sentiment of the review either negative, positive or neutral"]
    pros: Annotated[Optional[list[str]], "Write down all the pros inside a list"]
    cons: Annotated[Optional[list[str]], "Write down all the cons inside a list"]
    name: Annotated[Optional[str], "Write the name of the reviewer"]
    

structured_model = model.with_structured_output(Review)

result = structured_model.invoke("""I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful
                                 
Review by Nitish Singh
""")

print(result)
print(result['name'])

{'key_themes': ['performance', 'battery life', 'camera quality', 'S-Pen functionality', 'user interface', 'price'], 'summary': "The Samsung Galaxy S24 Ultra is a powerful device with a fast processor, excellent battery life, and a high-quality 200MP camera. While the S-Pen is a useful addition, the phone's size and weight, pre-installed bloatware, and high price point are notable drawbacks.", 'sentiment': 'pos', 'pros': ['Snapdragon 8 Gen 3 processor ensures fast performance for gaming, multitasking, and photo editing.', '5000mAh battery provides all-day usage, complemented by 45W fast charging.', 'The 200MP camera excels, especially in night mode, capturing detailed and vibrant low-light photos.', '100x zoom is functional for distant objects, with usable quality up to 30x.', 'S-Pen integration offers convenience for note-taking and sketching.'], 'cons': ["The phone's weight and size can make one-handed use uncomfortable.", "Pre-installed bloatware from Samsung's One UI duplicates func

PYDANTIC

In [26]:
from pydantic import BaseModel, Field

class Review(BaseModel):

    key_themes: list[str] = Field(description="Write down all the key themes discussed in the review in a list")
    summary: str = Field(description="A brief summary of the review")
    sentiment: Literal["pos", "neg"] = Field(description="Return sentiment of the review either negative, positive or neutral")
    pros: Optional[list[str]] = Field(default=None, description="Write down all the pros inside a list")
    cons: Optional[list[str]] = Field(default=None, description="Write down all the cons inside a list")
    name: Optional[str] = Field(default=None, description="Write the name of the reviewer")
    

structured_model = model.with_structured_output(Review)

result = structured_model.invoke("""I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful
                                 
Review by Nitish Singh
""")

print(result)

key_themes=['performance', 'battery life', 'camera quality', 'S-Pen functionality', 'design and ergonomics', 'software experience', 'price'] summary="The Samsung Galaxy S24 Ultra is a powerful smartphone with an excellent processor, impressive 200MP camera, and long battery life. While the S-Pen is a nice addition and the zoom is capable, the phone's size and weight, pre-installed bloatware, and high price point are notable drawbacks." sentiment='pos' pros=['Snapdragon 8 Gen 3 processor for fast performance', 'Long-lasting 5000mAh battery with 45W fast charging', 'Excellent 200MP camera with great night mode and usable zoom', 'S-Pen integration for note-taking and sketching', 'Stunning night mode photography', '100x zoom capable for distant objects'] cons=['Heavy and large, making one-handed use uncomfortable', "Includes bloatware from Samsung's One UI", 'Expensive at $1,300', 'Zoom quality degrades significantly beyond 30x'] name='Nitish Singh'


JSON

In [27]:
json_schema = {
  "title": "Review",
  "type": "object",
  "properties": {
    "key_themes": {
      "type": "array",
      "items": {
        "type": "string"
      },
      "description": "Write down all the key themes discussed in the review in a list"
    },
    "summary": {
      "type": "string",
      "description": "A brief summary of the review"
    },
    "sentiment": {
      "type": "string",
      "enum": ["pos", "neg"],
      "description": "Return sentiment of the review either negative, positive or neutral"
    },
    "pros": {
      "type": ["array", "null"],
      "items": {
        "type": "string"
      },
      "description": "Write down all the pros inside a list"
    },
    "cons": {
      "type": ["array", "null"],
      "items": {
        "type": "string"
      },
      "description": "Write down all the cons inside a list"
    },
    "name": {
      "type": ["string", "null"],
      "description": "Write the name of the reviewer"
    }
  },
  "required": ["key_themes", "summary", "sentiment"]
}


structured_model = model.with_structured_output(json_schema)

result = structured_model.invoke("""I recently upgraded to the Samsung Galaxy S24 Ultra, and I must say, it’s an absolute powerhouse! The Snapdragon 8 Gen 3 processor makes everything lightning fast—whether I’m gaming, multitasking, or editing photos. The 5000mAh battery easily lasts a full day even with heavy use, and the 45W fast charging is a lifesaver.

The S-Pen integration is a great touch for note-taking and quick sketches, though I don't use it often. What really blew me away is the 200MP camera—the night mode is stunning, capturing crisp, vibrant images even in low light. Zooming up to 100x actually works well for distant objects, but anything beyond 30x loses quality.

However, the weight and size make it a bit uncomfortable for one-handed use. Also, Samsung’s One UI still comes with bloatware—why do I need five different Samsung apps for things Google already provides? The $1,300 price tag is also a hard pill to swallow.

Pros:
Insanely powerful processor (great for gaming and productivity)
Stunning 200MP camera with incredible zoom capabilities
Long battery life with fast charging
S-Pen support is unique and useful
                                 
Review by Nitish Singh
""")

print(result)

{'key_themes': ['Performance', 'Battery Life', 'Camera Quality', 'S-Pen Functionality', 'Design and Ergonomics', 'Software Bloatware', 'Price'], 'summary': 'The Samsung Galaxy S24 Ultra is a powerful device with an excellent camera, long battery life, and fast charging. However, it is bulky, comes with pre-installed bloatware, and is quite expensive.', 'sentiment': 'pos', 'pros': ['Snapdragon 8 Gen 3 processor ensures lightning-fast performance', '5000mAh battery easily lasts a full day with heavy use', '45W fast charging is very convenient', '200MP camera captures stunning images, especially in low light', '100x zoom is functional for distant objects', 'S-Pen is useful for note-taking and sketching'], 'cons': ['Heavy and large, making one-handed use uncomfortable', 'One UI includes bloatware with redundant Samsung apps', 'High price tag of $1,300'], 'name': 'Nitish Singh'}
